# OTT Customer Churn Prediction

This project develops an end-to-end machine learning pipeline to predict
customer churn for an OTT streaming platform.

The workflow covers data exploration, preprocessing, class imbalance,
Random Forest modeling, hyperparameter optimization, model evaluation,
and deployment preparation.

Final deployed model:
- Test Accuracy: 91.2%
- Churn Precision: 67.4%
- Churn Recall: 66.7%
- Churn F1: 67.1%

In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [6]:
%matplotlib inline

## Dataset

The dataset used in this project was sourced from Kaggle.

**Source:** [Churn Modelling for OTT Platforms](https://www.kaggle.com/datasets/santhoshvr97/ott-chrun-modeling-ott)

To reproduce this notebook:

1. Download the dataset from Kaggle.
2. Create a folder named `data`.
3. Place `churn_1.csv` inside the `data` folder.

In [70]:
from pathlib import Path

data_path = Path("data/churn_1.csv")

if not data_path.exists():
    raise FileNotFoundError(
        "Dataset not found. Download churn_1.csv from the Kaggle link "
        "provided in the README and place it inside the data/ folder."
    )

df = pd.read_csv(data_path)
df.head()

FileNotFoundError: Dataset not found. Download churn_1.csv from the Kaggle link provided in the README and place it inside the data/ folder.

In [8]:
from autoviz.AutoViz_Class import AutoViz_Class
AV = AutoViz_Class()

Imported v0.1.905. Please call AutoViz in this sequence:
    AV = AutoViz_Class()
    %matplotlib inline
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)


In [9]:
dft=AV.AutoViz(
    df
)

Shape of your Data Set loaded: (2000, 16)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
    Number of Numeric Columns =  4
    Number of Integer-Categorical Columns =  6
    Number of String-Categorical Columns =  0
    Number of Factor-Categorical Columns =  0
    Number of String-Boolean Columns =  3
    Number of Numeric-Boolean Columns =  1
    Number of Discrete String Columns =  0
    Number of NLP String Columns =  0
    Number of Date Time Columns =  0
    Number of ID Columns =  1
    Number of Columns to Delete =  1
    16 Predictors classified...
        2 variable(s) removed since they were ID or low-information variables
        List of variables removed: ['phone_no', 'year']
To fix these data quality issues in the dataset

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
year,int64,0.000000,0,2020.000000,2020.000000,Possible Zero-variance or low information colum: drop before modeling step.
customer_id,int64,0.000000,99,100198.000000,999961.000000,No issue
phone_no,object,0.000000,100,,,Possible ID column: drop before modeling step.
gender,object,1.200000,0,,,"24 missing values. Impute them with mean, median, mode, or a constant value such as 123., Mixed dtypes: has 2 different data types: object, float,"
age,int64,0.000000,3,18.000000,82.000000,Column has 63 outliers greater than upper bound (62.00) or lower than lower bound(14.00). Cap them or remove them.
no_of_days_subscribed,int64,0.000000,10,1.000000,243.000000,Column has 11 outliers greater than upper bound (208.00) or lower than lower bound(-8.00). Cap them or remove them.
multi_screen,object,0.000000,0,,,No issue
mail_subscribed,object,0.000000,0,,,No issue
weekly_mins_watched,float64,0.000000,NA,0.000000,526.200000,Column has 18 outliers greater than upper bound (484.37) or lower than lower bound(58.52). Cap them or remove them.
minimum_daily_mins,float64,0.000000,NA,0.000000,20.000000,Column has 25 outliers greater than upper bound (17.40) or lower than lower bound(3.00). Cap them or remove them.


Number of All Scatter Plots = 10
All Plots done
Time to run AutoViz = 8 seconds 

 ###################### AUTO VISUALIZATION Completed ########################


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   year                    2000 non-null   int64  
 1   customer_id             2000 non-null   int64  
 2   phone_no                2000 non-null   object 
 3   gender                  1976 non-null   object 
 4   age                     2000 non-null   int64  
 5   no_of_days_subscribed   2000 non-null   int64  
 6   multi_screen            2000 non-null   object 
 7   mail_subscribed         2000 non-null   object 
 8   weekly_mins_watched     2000 non-null   float64
 9   minimum_daily_mins      2000 non-null   float64
 10  maximum_daily_mins      2000 non-null   float64
 11  weekly_max_night_mins   2000 non-null   int64  
 12  videos_watched          2000 non-null   int64  
 13  maximum_days_inactive   1972 non-null   float64
 14  customer_support_calls  2000 non-null   

In [11]:
df.describe()

,year,customer_id,age,no_of_days_subscribed,weekly_mins_watched,minimum_daily_mins,maximum_daily_mins,weekly_max_night_mins,videos_watched,maximum_days_inactive,customer_support_calls,churn
count,2000.0,2000.00000,2000.00000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,1972.000000,2000.000000,1965.000000
mean,2020.0,554887.16250,38.69050,99.750000,270.178425,10.198700,30.620780,100.415500,4.482500,3.250507,1.547000,0.133333
std,0.0,261033.69218,10.20641,39.755386,80.551627,2.785519,9.129165,19.529454,2.487728,0.809084,1.315164,0.340021
min,2020.0,100198.00000,18.00000,1.000000,0.000000,0.000000,0.000000,42.000000,0.000000,0.000000,0.000000,0.000000
25%,2020.0,328634.75000,32.00000,73.000000,218.212500,8.400000,24.735000,87.000000,3.000000,3.000000,1.000000,0.000000
50%,2020.0,567957.50000,37.00000,99.000000,269.925000,10.200000,30.590000,101.000000,4.000000,3.000000,1.000000,0.000000
75%,2020.0,773280.25000,44.00000,127.000000,324.675000,12.000000,36.797500,114.000000,6.000000,4.000000,2.000000,0.000000
max,2020.0,999961.00000,82.00000,243.000000,526.200000,20.000000,59.640000,175.000000,19.000000,6.000000,9.000000,1.000000


In [12]:
#Columns not needed for predictions
drop_cols = ["customer_id", "phone_no", "year"]

df = df.drop(columns=drop_cols, errors="ignore")


In [13]:
df.columns.tolist()

['gender',
 'age',
 'no_of_days_subscribed',
 'multi_screen',
 'mail_subscribed',
 'weekly_mins_watched',
 'minimum_daily_mins',
 'maximum_daily_mins',
 'weekly_max_night_mins',
 'videos_watched',
 'maximum_days_inactive',
 'customer_support_calls',
 'churn']

In [14]:
#Convert all yes and no to 0 and 1
yn_map = {"yes": 1, "no": 0, "Yes": 1, "No": 0, True: 1, False: 0}

for col in ["multi_screen", "mail_subscribed"]:
    df[col] = df[col].astype(str).str.strip().map(yn_map)
    
print(df[["multi_screen", "mail_subscribed"]].head())


   multi_screen  mail_subscribed
0        0              0       
1        0              0       
2        0              0       
3        0              1       
4        0              0       


In [15]:
df = df.dropna(subset=["churn"]).copy()
df["churn"] = df["churn"].astype(int)
print(df["churn"].value_counts())

churn
0    1703
1     262
Name: count, dtype: int64


In [16]:
corr = df.select_dtypes(include="number").corr()

#Heatmap
#Correlation
corr = df.corr(numeric_only=True)
plt.figure(figsize=(12,10))
sns.heatmap(corr, annot=True)
plt.tight_layout()

In [17]:
df.churn

0       0
1       0
2       1
3       0
4       0
       ..
1992    0
1996    0
1997    0
1998    0
1999    1
Name: churn, Length: 1965, dtype: int32

In [18]:
#Independet features
X = df.drop(columns=["churn"])
X.head()

,gender,age,no_of_days_subscribed,multi_screen,mail_subscribed,weekly_mins_watched,minimum_daily_mins,maximum_daily_mins,weekly_max_night_mins,videos_watched,maximum_days_inactive,customer_support_calls
0,Female,36,62,0,0,148.35,12.2,16.81,82,1,4.0,1
1,Female,39,149,0,0,294.45,7.7,33.37,87,3,3.0,2
2,Female,65,126,0,0,87.30,11.9,9.89,91,1,4.0,5
3,Female,24,131,0,1,321.30,9.5,36.41,102,4,3.0,3
4,Female,40,191,0,0,243.00,10.9,27.54,83,7,3.0,1


In [19]:
#Target
y = df["churn"].astype(int)

In [20]:
#Numerical Columns
nc=['age',
 'no_of_days_subscribed',
 'multi_screen',
 'mail_subscribed',
 'weekly_mins_watched',
 'minimum_daily_mins',
 'maximum_daily_mins',
 'weekly_max_night_mins',
 'videos_watched',
 'maximum_days_inactive',
 'customer_support_calls',]

#Categorical Columns

cc=['gender']

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.33,
    random_state=42,
    stratify=y   # important for churn imbalance
)

In [22]:
#Pipelining
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder #Feature Scaling
from sklearn.compose import ColumnTransformer

In [23]:
#Automatiuon of Feature engineering
#Numerical pipeline
num_pipeline=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)
#Categorical Pipeline
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
    

In [24]:
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, nc),   # nc = numeric columns list
    ("cat", cat_pipeline, cc)    # cc = categorical columns list
])

In [25]:
y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)


(churn
 0    0.867021
 1    0.132979
 Name: proportion, dtype: float64,
 churn
 0    0.865948
 1    0.134052
 Name: proportion, dtype: float64)

In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
# Model Training Automation
pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

In [27]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [28]:
param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 6, 10, 14, 18],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2"]
}

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=40,        # 40 candidates 
    cv=3,             # 3 folds instead of 5
    scoring="f1",
    n_jobs=-1,
    verbose=0,
    random_state=42
)

search.fit(X_train, y_train)

best_pipe_r = search.best_estimator_
print("Best CV F1:", search.best_score_)
print("Best params:", search.best_params_)

Fitting 3 folds for each of 40 candidates, totalling 120 fits
Best CV F1: 0.6573588949749941
Best params: {'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': None}


In [29]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = best_pipe_r.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))


[[534  28]
 [ 29  58]]
              precision    recall  f1-score   support

           0      0.948     0.950     0.949       562
           1      0.674     0.667     0.671        87

    accuracy                          0.912       649
   macro avg      0.811     0.808     0.810       649
weighted avg      0.912     0.912     0.912       649



In [30]:
#Using GridSearchCV
from sklearn.model_selection import GridSearchCV

small_grid = {
    "model__n_estimators":    [300, 500],
    "model__max_depth":       [None, 8, 12],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf":  [1, 2],
    "model__max_features":    ["sqrt"],
}

grid = GridSearchCV(
    pipe,
    param_grid=small_grid,
    cv=5,                # 5 folds → 24 * 5 = 120 fits
    scoring="recall",    # goal is catching churn
    n_jobs=-1,
    verbose=0
)

grid.fit(X_train, y_train)

best_pipe_grid = grid.best_estimator_
print("Best CV recall:", grid.best_score_)
print("Best params:", grid.best_params_)


Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best CV recall: 0.6171428571428571
Best params: {'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 10, 'model__n_estimators': 300}


In [31]:
y_pred_g = best_pipe_grid.predict(X_test)

print(confusion_matrix(y_test, y_pred_g))
print(classification_report(y_test, y_pred_g, digits=3))

[[527  35]
 [ 32  55]]
              precision    recall  f1-score   support

           0      0.943     0.938     0.940       562
           1      0.611     0.632     0.621        87

    accuracy                          0.897       649
   macro avg      0.777     0.785     0.781       649
weighted avg      0.898     0.897     0.898       649



# Final Model Selection

Although both RandomizedSearchCV and GridSearchCV were evaluated,
the RandomizedSearchCV model generalized better on the held-out test set.

Therefore, `best_pipe_r` was selected as the final model for deployment.

In [32]:
import joblib

In [33]:
joblib.dump(
    best_pipe_r,
    "churn_model.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [68]:
# Verify saved model

loaded_model = joblib.load(
    "churn_model.joblib"
)

sample_customer = X_test.iloc[[0]]

prediction = loaded_model.predict(
    sample_customer
)[0]

probability = loaded_model.predict_proba(
    sample_customer
)[0][1]

print("Actual:", y_test.iloc[0])
print("Prediction:", prediction)
print(
    "Churn probability:",
    round(probability * 100, 2),
    "%"
)

Actual: 0
Prediction: 0
Churn probability: 8.49 %


# Conclusion

The selected Random Forest pipeline achieved 91.2% accuracy and a
67.1% F1 score for the minority churn class.

The complete preprocessing and classification pipeline was serialized
with Joblib and deployed as an interactive Streamlit application.

Live application:
https://ott-churn-prediction-yb.streamlit.app